<a href="https://colab.research.google.com/github/manasah3/CS598-DLH/blob/final_project_dlhCS498/colab_dreamtdata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS 598 DL4H — WatchSleepNet Colab POC

This notebook validates the Colab ↔ GitHub ↔ Google Drive workflow for the WatchSleepNet replication project.

**Branch:** `colab-poc`  
**Repo:** [cs598-DLH-WatchSleepNet](https://github.com/jananij2/cs598-DLH-WatchSleepNet)

## 1. Install Dependencies

In [1]:
# Install PyHealth from source (use this during active contribution work)
!pip install git+https://github.com/sunlabuiuc/PyHealth.git -q

# Other common dependencies
!pip install numpy pandas scikit-learn matplotlib -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import pyhealth
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"PyHealth version: {pyhealth.__version__}")
print("All imports successful.")

PyHealth version: 2.0.0
All imports successful.


## 2. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

# Base path — adjust if your Drive folder structure differs
DRIVE_BASE = '/content/drive/MyDrive/cs598-watchsleepnet'

PATHS = {
    'raw':       os.path.join(DRIVE_BASE, 'raw'),
    'processed': os.path.join(DRIVE_BASE, 'processed'),
    'models':    os.path.join(DRIVE_BASE, 'models'),
    'results':   os.path.join(DRIVE_BASE, 'results'),
}

for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f"{name:12s} → {path}")

raw          → /content/drive/MyDrive/cs598-watchsleepnet/raw
processed    → /content/drive/MyDrive/cs598-watchsleepnet/processed
models       → /content/drive/MyDrive/cs598-watchsleepnet/models
results      → /content/drive/MyDrive/cs598-watchsleepnet/results


## 3. Dataset Implementation (dreamt.py)


In [5]:
import os
import pandas as pd
from typing import Dict, List, Any, Optional
from pyhealth.datasets import BaseDataset

class DREAMTDataset(BaseDataset):
    """Dataset for Real-time sleep stage Estimation (DREAMT)."""

    def __init__(self, root: str, dev: bool = True, **kwargs):
        import time
        # We use a unique name to ensure we don't load a broken old cache
        unique_name = f"DREAMT_{int(time.time())}"
        super(DREAMTDataset, self).__init__(
            dataset_name=unique_name,
            root=root,
            tables=["whole_df"],
            dev=dev,
            **kwargs
        )

    def parse_basic_info(self) -> Dict[str, Dict[str, Any]]:
        """Parses demographics from participant_info.csv."""
        info_path = os.path.join(self.root, "participant_info.csv")
        df = pd.read_csv(info_path)

        # Standardize column names to uppercase to match your file
        df.columns = df.columns.str.strip().str.upper()
        print(f"DEBUG: Successfully mapped headers: {df.columns.tolist()}")

        data = {}
        for _, row in df.iterrows():
            # Your SID already looks like 'S002', so we use it directly
            sid = str(row["SID"]).strip()
            data[sid] = {
                "age": row.get("AGE", 0),
                "gender": row.get("GENDER", "Unknown"),
                "ahi": row.get("AHI", 0)
            }
        return data

    def parse_tables(self) -> Dict[str, Dict[str, List]]:
        """Parses the time-series signal CSVs."""
        patients = self.parse_basic_info()
        data_dir = os.path.join(self.root, "data_64Hz")

        final_patients = {}

        for sid in list(patients.keys()):
            file_path = os.path.join(data_dir, f"{sid}_whole_df.csv")

            if os.path.exists(file_path):
                # Load the 133MB file
                df = pd.read_csv(file_path)

                # Standardize signal columns
                df.columns = df.columns.str.strip()

                final_patients[sid] = {
                    "patient_id": sid,
                    "age": patients[sid]["age"],
                    "gender": patients[sid]["gender"],
                    # Map exactly to the columns in S00x_whole_df.csv
                    "ibi": df["IBI"].tolist(),
                    "label": df["Sleep_Stage"].tolist()
                }

        self.patients = final_patients
        return final_patients

## 4. Task Implementation (sleep_staging_dreamt.py)

In [6]:
from typing import Dict, List

def sleep_staging_dreamt_fn(patient: Dict) -> List[Dict]:
    """Task function to window IBI data into 30-second epochs.

    Window Logic: 30 seconds @ 64Hz = 1920 samples.
    """
    samples = []
    # Note: We use the keys we defined in the Dataset class
    ibi = patient.get("ibi", [])
    labels = patient.get("label", [])

    window_size = 1920
    label_map = {"W": 0, "N1": 1, "N2": 2, "N3": 3, "R": 4}

    for i in range(0, len(ibi) - window_size, window_size):
        epoch_ibi = ibi[i : i + window_size]
        epoch_label = labels[i + window_size - 1]

        if epoch_label in label_map:
            samples.append({
                "patient_id": patient["patient_id"],
                "feature": epoch_ibi,
                "label": label_map[epoch_label]
            })
    return samples

# REQUIRED FOR PYHEALTH RUBRIC:
sleep_staging_dreamt_fn.task_name = "sleep_staging_dreamt"

## 5.  Execution and Verification


In [7]:
print("--- Step 1: Loading Dataset ---")
dataset = DREAMTDataset(root=PATHS['raw'], dev=True)

# If PyHealth's internal loading failed, we force our parsed data in
if not hasattr(dataset, 'patients') or len(dataset.patients) == 0:
    print("Forcing manual parse...")
    dataset.patients = dataset.parse_tables()

print(f"✅ Success! {len(dataset.patients)} subjects ready.")

# 2. Apply Task
print("\n--- Step 2: Applying Task ---")
# We use the dataset's own logic to process the patients
samples = []
for sid, patient in dataset.patients.items():
    samples.extend(sleep_staging_dreamt_fn(patient))

print(f"✅ Total sleep epochs created: {len(samples)}")

if len(samples) > 0:
    print(f"Sample 0 Feature Length: {len(samples[0]['feature'])} (Expected: 1920)")
    print(f"Sample 0 Label: {samples[0]['label']}")

--- Step 1: Loading Dataset ---
Initializing DREAMT_1775103270 dataset from /content/drive/MyDrive/cs598-watchsleepnet/raw (dev mode: True)


INFO:pyhealth.datasets.base_dataset:Initializing DREAMT_1775103270 dataset from /content/drive/MyDrive/cs598-watchsleepnet/raw (dev mode: True)


No cache_dir provided. Using default cache dir: /root/.cache/pyhealth/9abe5e42-eaee-5262-8414-fda469929a2f


INFO:pyhealth.datasets.base_dataset:No cache_dir provided. Using default cache dir: /root/.cache/pyhealth/9abe5e42-eaee-5262-8414-fda469929a2f


Forcing manual parse...
DEBUG: Successfully mapped headers: ['SID', 'AGE', 'GENDER', 'BMI', 'OAHI', 'AHI', 'MEAN_SAO2', 'AROUSAL INDEX', 'MEDICAL_HISTORY', 'SLEEP_DISORDERS']
✅ Success! 5 subjects ready.

--- Step 2: Applying Task ---
✅ Total sleep epochs created: 3984
Sample 0 Feature Length: 1920 (Expected: 1920)
Sample 0 Label: 0


##6. WatchSleepNet Model Implementation (watchsleepnet.py)
This cell contains the actual neural network. We have added the TCN and Attention layers to match the paper's architecture exactly.


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Optional
from pyhealth.models import BaseModel

class ResidualBlock(nn.Module):
    """1D Residual Block for spatial feature extraction."""
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=5, stride=stride, padding=2)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=5, stride=1, padding=2)
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm1d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return F.relu(out)

class WatchSleepNet(BaseModel):
    """WatchSleepNet implementation following CHIL 2025 paper.

    Architecture:
    1. ResNet: Spatial feature extraction from IBI windows.
    2. TCN: Temporal convolutional network for long-range dependencies.
    3. Bi-LSTM + Attention: Sequential modeling and feature relevance.
    """
    def __init__(
        self,
        dataset: Optional[Any] = None,
        feature_size: int = 1920,
        hidden_dim: int = 256,
        **kwargs
    ):
        super(WatchSleepNet, self).__init__(dataset, **kwargs)

        # Stage 1: ResNet (Spatial)
        # Input: (Batch, 1, 1920) -> Output: (Batch, 256, 120)
        self.resnet = nn.Sequential(
            ResidualBlock(1, 64, stride=2),          # 1920 -> 960
            ResidualBlock(64, 128, stride=2),         # 960 -> 480
            ResidualBlock(128, 256, stride=2),        # 480 -> 240
            ResidualBlock(256, hidden_dim, stride=2), # 240 -> 120
        )

        # Stage 2: TCN (Temporal)
        # Dilated convolutions to capture dependencies
        self.tcn = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, dilation=2, padding=4)

        # Stage 3: Bi-LSTM + Attention
        self.lstm = nn.LSTM(hidden_dim, 128, batch_first=True, bidirectional=True)
        self.attention = nn.MultiheadAttention(embed_dim=256, num_heads=8, batch_first=True)

        # Final Classifier: 5 sleep stages
        self.fc = nn.Linear(256, 5)

    def forward(self, feature: torch.Tensor, label: torch.Tensor = None, **kwargs) -> Dict[str, torch.Tensor]:
        # 1. ResNet Spatial Extraction
        # Input: [Batch, 1920] -> [Batch, 1, 1920]
        x = feature.unsqueeze(1).float()
        x = self.resnet(x) # [Batch, 256, 120]

        # 2. TCN Temporal Extraction
        x = F.relu(self.tcn(x)) # [Batch, 256, 120]

        # 3. Bi-LSTM
        # Re-arrange for LSTM: [Batch, Sequence_Len, Hidden_Dim]
        x = x.transpose(1, 2) # [Batch, 120, 256]
        x, _ = self.lstm(x) # [Batch, 120, 256]

        # 4. Multi-head Attention
        # self-attention across the 120 time-steps
        attn_out, _ = self.attention(x, x, x)

        # 5. Global Pooling and Classify
        # We take the mean across the sequence dimension
        x = torch.mean(attn_out, dim=1)
        logits = self.fc(x)

        y_prob = torch.softmax(logits, dim=-1)

        loss = 0
        if label is not None:
            loss = F.cross_entropy(logits, label.long())

        return {"loss": loss, "y_prob": y_prob, "y_true": label}

##7. Model Verification and Training Prep

In [9]:
import torch
from torch.utils.data import DataLoader

# 1. Initialize the model
model = WatchSleepNet()

# 2. Convert samples to PyTorch DataLoader (Standard PyHealth/Torch way)
# For testing, we take a tiny batch of 4 samples
test_batch = samples[:4]
features = torch.stack([torch.tensor(s['feature']) for s in test_batch])
labels = torch.tensor([s['label'] for s in test_batch])

# 3. Run Forward Pass
output = model(feature=features, label=labels)

print(f"--- Model Verification ---")
print(f"Input batch shape: {features.shape}")
print(f"Output probability shape: {output['y_prob'].shape}") # Should be [4, 5]
print(f"Initial Loss: {output['loss'].item():.4f}")

if output['y_prob'].shape == (4, 5):
    print("✅ Model Architecture Verified!")

--- Model Verification ---
Input batch shape: torch.Size([4, 1920])
Output probability shape: torch.Size([4, 5])
Initial Loss: 1.6030
✅ Model Architecture Verified!


##8. Training loop

In [10]:
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# 1. SPLIT SAMPLES (80% Train, 20% Test)
indices = list(range(len(samples)))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

train_data = [samples[i] for i in train_idx]
test_data = [samples[i] for i in test_idx]

# 2. CREATE DATALOADERS
def collate_fn(batch):
    # PyHealth samples are dicts; we stack them into tensors
    features = torch.stack([torch.tensor(item['feature']).float() for item in batch])
    labels = torch.tensor([item['label'] for item in batch]).long()
    return features, labels

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, collate_fn=collate_fn)

# 3. SETUP TRAINING
device = "cuda" if torch.cuda.is_available() else "cpu"
model = WatchSleepNet()
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 4. TRAINING LOOP
print(f"Training WatchSleepNet on {device}...")
for epoch in range(1, 11): # 10 Epochs
    model.train()
    train_loss = 0
    for features, labels in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
        features, labels = features.to(device), labels.to(device)

        optimizer.zero_grad()
        output = model(feature=features, label=labels)
        loss = output['loss']

        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # 5. EVALUATION
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for features, labels in test_loader:
            features, labels = features.to(device), labels.to(device)
            output = model(feature=features)
            preds = torch.argmax(output['y_prob'], dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    print(f"Epoch {epoch} | Loss: {train_loss/len(train_loader):.4f} | Test Acc: {acc:.4f}")

print("\n✅ Training Complete!")

Training WatchSleepNet on cuda...


Epoch 1:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1 | Loss: 1.2931 | Test Acc: 0.4944


Epoch 2:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 2 | Loss: 1.2513 | Test Acc: 0.5508


...

Epoch 8 | Loss: 1.1412 | Test Acc: 0.5809


Epoch 9:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 9 | Loss: 1.1044 | Test Acc: 0.5445


Epoch 10:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 10 | Loss: 1.1218 | Test Acc: 0.5759

✅ Training Complete!
